<a href="https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriNehe/Flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
!pip install duckdb --quiet
import duckdb, os
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

features = con.sql("""
    SELECT c.content_hash_id, c.word_count, c.char_count,
           DATE_DIFF('day', c.content_created_date, DATE '2025-12-01') AS age_days_dec,
           f.gsc_impressions AS impressions_dec, f.gsc_clicks AS clicks_dec,
           f.gsc_sum_position AS position_dec
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
    JOIN (
        SELECT content_hash_id, SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks, AVG(gsc_sum_position) AS gsc_sum_position
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-12/data_0.parquet'
        GROUP BY content_hash_id
    ) f ON c.content_hash_id = f.content_hash_id
    WHERE c.is_published IS TRUE
""").df()

# Handle missing values — fill numeric gaps with 0 (no signal = treated as no activity)
features_filled = features.fillna({
    "word_count": 0, "char_count": 0,
    "impressions_dec": 0, "clicks_dec": 0, "position_dec": 0
})

print("Feature vector shape:", features_filled.shape)
print(features_filled.dtypes)
print(features_filled.isnull().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (237435, 7)
content_hash_id     object
word_count           Int64
char_count           Int64
age_days_dec         int64
impressions_dec    float64
clicks_dec         float64
position_dec       float64
dtype: object
content_hash_id    0
word_count         0
char_count         0
age_days_dec       0
impressions_dec    0
clicks_dec         0
position_dec       0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
print("word_count — meaning: number of words in the published content body")
print("  Missing: filled with 0 (rare, only for rows with no content record)")
print("  Available-when: exists at publish time, well before Dec 2025 decision point\n")

print("char_count — meaning: character count of the content body")
print("  Missing: filled with 0, same as word_count")
print("  Available-when: exists at publish time\n")

print("age_days_dec — meaning: days between content_created_date and Dec 1, 2025")
print("  Missing: not possible — computed directly from content_created_date")
print("  Available-when: known as soon as the page is created, always before decision point\n")

print("impressions_dec — meaning: total GSC impressions in December 2025")
print("  Missing: filled with 0 (no GSC data or zero impressions that month)")
print("  Available-when: known by end of December 2025, before the March 2026 label exists\n")

print("clicks_dec — meaning: total GSC clicks in December 2025")
print("  Missing: filled with 0, same logic as impressions_dec")
print("  Available-when: known by end of December 2025\n")

print("position_dec — meaning: average search ranking position in December 2025")
print("  Missing: filled with 0 (treated as no ranking data — a limitation, not a true position)")
print("  Available-when: known by end of December 2025")

word_count — meaning: number of words in the published content body
  Missing: filled with 0 (rare, only for rows with no content record)
  Available-when: exists at publish time, well before Dec 2025 decision point

char_count — meaning: character count of the content body
  Missing: filled with 0, same as word_count
  Available-when: exists at publish time

age_days_dec — meaning: days between content_created_date and Dec 1, 2025
  Missing: not possible — computed directly from content_created_date
  Available-when: known as soon as the page is created, always before decision point

impressions_dec — meaning: total GSC impressions in December 2025
  Missing: filled with 0 (no GSC data or zero impressions that month)
  Available-when: known by end of December 2025, before the March 2026 label exists

clicks_dec — meaning: total GSC clicks in December 2025
  Missing: filled with 0, same logic as impressions_dec
  Available-when: known by end of December 2025

position_dec — meaning: av

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# Attack own features: check for label-derived columns, future windows, product flags
feature_cols = list(features_filled.columns)

leak_check_names = [c for c in feature_cols if "march" in c.lower() or "target" in c.lower() or "declin" in c.lower()]
print("Suspicious column names found:", leak_check_names if leak_check_names else "None")

# Confirm no feature uses data later than Dec 2025
future_window_check = con.sql("""
    SELECT MAX(report_date) AS latest_feature_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-12/data_0.parquet'
""").df()
print("\nLatest date used in features:", future_window_check)
print("Confirms all features stop at December 2025 — nothing from Jan/Feb/March 2026 leaked in.")

# Check age_days_dec isn't accidentally computed relative to a future date
print("\nage_days_dec computed relative to: 2025-12-01 (fixed decision point)")
print("Sample age_days_dec range:", features_filled["age_days_dec"].min(), "to", features_filled["age_days_dec"].max())

# No product/client flags accidentally included
product_flag_check = [c for c in feature_cols if "flag" in c.lower() or "client_id" in c.lower() or "provider" in c.lower()]
print("\nProduct/client flags in feature set:", product_flag_check if product_flag_check else "None — only content_hash_id kept, and only for joining")

Suspicious column names found: None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Latest date used in features:   latest_feature_date
0          2025-12-31
Confirms all features stop at December 2025 — nothing from Jan/Feb/March 2026 leaked in.

age_days_dec computed relative to: 2025-12-01 (fixed decision point)
Sample age_days_dec range: -65 to 368

Product/client flags in feature set: None — only content_hash_id kept, and only for joining


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
print("Excluded fields and why:\n")

print("- sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other")
print("  Why: sparse/mostly null across the client base — including them would silently drop")
print("  or bias rows where AI-referral tracking isn't set up.\n")

print("- scroll_events")
print("  Why: depends on ga4_data_available, which is not true for every client —")
print("  same sparsity risk as the ai_* columns.\n")

print("- content_hash_id, client_hash_id, report_date")
print("  Why: kept only as context/join keys, never used as model features —")
print("  they carry no predictive signal about decline risk.\n")

print("- gsc_sum_position (raw daily), gsc_clicks (raw daily)")
print("  Why: used only in aggregated form (position_dec, clicks_dec as Dec-month sums/averages)")
print("  rather than daily granularity, to match the monthly decision cadence of the lane.\n")

print("- Any column referencing March 2026 or later")
print("  Why: would leak the label window into the features — confirmed absent in Section 3.")

Excluded fields and why:

- sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other
  Why: sparse/mostly null across the client base — including them would silently drop
  or bias rows where AI-referral tracking isn't set up.

- scroll_events
  Why: depends on ga4_data_available, which is not true for every client —
  same sparsity risk as the ai_* columns.

- content_hash_id, client_hash_id, report_date
  Why: kept only as context/join keys, never used as model features —
  they carry no predictive signal about decline risk.

- gsc_sum_position (raw daily), gsc_clicks (raw daily)
  Why: used only in aggregated form (position_dec, clicks_dec as Dec-month sums/averages)
  rather than daily granularity, to match the monthly decision cadence of the lane.

- Any column referencing March 2026 or later
  Why: would leak the label window into the features — confirmed absent in Section 3.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.